Awesome 👍 Let’s extend this into a **multi-agent LangGraph setup with tools** (like your `math_server`).

We’ll design a **controlled agent-to-agent communication** flow:

* **User → Planner Agent → Math Agent → Summarizer Agent → Output**

---

## 1️⃣ Config: `agents.yaml`

```yaml
agents:
  planner:
    llm: openai
    tools: []   # Planner only decides who to call

  math_agent:
    llm: openai
    mcp_servers:
      math_server:
        command: "python"
        args: ["math_server.py"]
        transport: "stdio"

  summarizer:
    llm: openai
    tools: []   # Just summarizes results
```

---

## 2️⃣ Math Server (`math_server.py`)

Already fixed earlier ✅

```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Math")

@mcp.tool()
def add(a: int, b: int) -> int:
    return a + b

@mcp.tool()
def multiply(a: int, b: int) -> int:
    return a * b

if __name__ == "__main__":
    mcp.run(transport="stdio")
```

---

## 3️⃣ LangGraph Flow (`main.py`)

```python
import asyncio
from langgraph.graph import StateGraph, END
from agent_factory import load_config, build_agent   # from your earlier code

# Define graph state
class GraphState(dict):
    pass

async def main():
    config = load_config("config/agents.yaml")

    # Build agents
    planner = await build_agent("planner", config)
    math_agent = await build_agent("math_agent", config)
    summarizer = await build_agent("summarizer", config)

    # Graph workflow
    workflow = StateGraph(GraphState)

    # Add nodes (agents as functions)
    workflow.add_node("planner", lambda state: planner.ainvoke(state))
    workflow.add_node("math_agent", lambda state: math_agent.ainvoke(state))
    workflow.add_node("summarizer", lambda state: summarizer.ainvoke(state))

    # Edges (who can talk to whom)
    workflow.set_entry_point("planner")
    workflow.add_edge("planner", "math_agent")
    workflow.add_edge("math_agent", "summarizer")
    workflow.set_finish_point("summarizer")

    # Compile and run
    app = workflow.compile()

    result = await app.ainvoke({"messages": [{"role": "user", "content": "What is 12 * 8?"}]})
    print("Final Output:", result)

if __name__ == "__main__":
    asyncio.run(main())
```

---

## 🔗 Flow

1. **User** asks: *“What is 12 \* 8?”*
2. **Planner Agent** → decides math is needed.
3. **Math Agent** → calls `math_server` tool → returns `96`.
4. **Summarizer Agent** → formats: *“The answer is 96.”*
5. **Final Output** → back to user.

---

## 🎯 Why This Works

* **Agent-to-agent communication** is controlled via graph edges.
* **No infinite loops** (Planner can only talk to Math, Math only to Summarizer).
* **Tools (MCP servers)** integrate smoothly into the agent workflow.

---

👉 Do you want me to **visualize this graph** (LangGraph supports `.draw()`) so you can literally see the Planner → Math Agent → Summarizer pipeline?
